# Gerando buffer de 300 metros e gerando tiles

## Etapa 1

In [11]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

crs = 4326
buffer_size = 0.002700003 # Em graus pelo crs ser 4326 (WGS84 geográfico)
pixel_reduction = 0.01 # Em graus pelo crs ser 4326 (WGS84 geográfico)

# Importando
tile_grid_gdf = gpd.read_file('inputs/mesh_z11/tile_grid_z11_BR.shp')
stations_df = pd.read_csv('inputs/stations/Monitoramento_QAr_BR.csv')

# Convertendo estações para geodataframe
stations_df.loc[:, 'geometry'] = stations_df[['LATITUDE', 'LONGITUDE']].apply(
    lambda x: Point(x['LONGITUDE'], x['LATITUDE']), axis=1)

stations_gdf = gpd.GeoDataFrame(
    stations_df[['LATITUDE', 'LONGITUDE', 'geometry']], 
    crs=4326,
    geometry='geometry'
)

# Criando buffer de 0,002700003 (aproximadamente 300m no nível do equador)
stations_gdf_buff = stations_gdf.buffer(buffer_size)

# Plotando para ver se está tudo certo
stations_gdf_buff.explore()

/tmp/ipykernel_392641/1733778247.py:24: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  stations_gdf_buff = stations_gdf.buffer(buffer_size)


In [12]:
# Selecionando as células de grade que possuem estação
intersection_s = tile_grid_gdf.sjoin(
    gpd.GeoDataFrame(
        dict(geometry=stations_gdf_buff.values),
        geometry='geometry',
        crs=4326
    ),
    how='left',
    predicate='intersects'
)

# Selecionando somente aqueles
intersection_s = intersection_s.loc[~intersection_s.index_right.isna()]

# Removendo colunas desnecessárias e removendo duplicados
intersection_s = intersection_s[['zoom', 'xtile', 'ytile', 'geometry']]
intersection_s = intersection_s.loc[~intersection_s.index.duplicated(keep='first')]
intersection_s

/home/nobre/Notebooks/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:2560: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  return geopandas.sjoin(


,zoom,xtile,ytile,geometry
2439,11,729,1002,"POLYGON ((-51.85547 3.68886, -51.85547 3.86425..."
2440,11,730,1002,"POLYGON ((-51.67969 3.68886, -51.67969 3.86425..."
2898,11,674,1004,"POLYGON ((-61.52344 3.33795, -61.52344 3.51342..."
3673,11,678,1007,"POLYGON ((-60.82031 2.81137, -60.82031 2.98693..."
5727,11,676,1015,"POLYGON ((-61.17188 1.40611, -61.17188 1.58183..."
...,...,...,...,...
54098,11,731,1203,"POLYGON ((-51.50391 -30.14513, -51.50391 -29.9..."
54099,11,732,1203,"POLYGON ((-51.32812 -30.14513, -51.32812 -29.9..."
56398,11,718,1212,"POLYGON ((-53.78906 -31.50363, -53.78906 -31.3..."
56655,11,718,1213,"POLYGON ((-53.78906 -31.65338, -53.78906 -31.5..."


In [16]:
# Reduzindo levemente o tamanho do pixel, para evitar que pegue área de outros
intersection_s = gpd.GeoDataFrame(intersection_s, geometry='geometry', crs=crs)

intersection_s['geometry'] = intersection_s.buffer(distance=-pixel_reduction)

/tmp/ipykernel_392641/3636611833.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersection_s['geometry'] = intersection_s.buffer(distance=-pixel_reduction)


In [17]:
# Salvando tile por tile em arquivos separados
# Essa passo foi feito somente para ter a possibilidade de paralelizar a aquisição das tiles
for idx, row in enumerate(intersection_s.itertuples()):
    gpd.gpd.GeoDataFrame(
        dict(
            zoom=[row.zoom],
            xtile=[row.xtile],
            ytile=[row.ytile],
            geometry=[row.geometry]
        ), geometry='geometry', crs=4326).to_file(f'outputs/tile_{idx}.gpkg')

<span style="color:red"><b>Observação:</b></span> O resultado dessa saída foi usado para obter as tiles. Como a tarefa é computacionalmente custosa, ela teve que ser rodada em dois locais e em mais três etapas.

## Etapa 2 - Agrupamento por máscara e agregação temporal

Os arquivos em "outputs" foram movidos para um computador local, que serviram para selecionar as vias com informações agregadas a serem utilizadas no processo de cômputo da representação espacial em etapas seguintes.